Phase 8: drift monitoring. Did the data, and the models' scores, move between the window the model learned from and the
window it was judged on? Writes the public drift page for GitHub Pages (docs/) and the tables for the README (reports/).

Feature drift: reference = the TRAIN window (Sep 1-6, what XGBoost was fitted on), current = the TEST window (Sep 9-10).
Prediction drift: reference = VALIDATION scores (Sep 7-8, where the alert lines were set), current = TEST scores. Both are
out-of-sample, so this is a fair comparison (train scores would be in-sample and look better than they are).
The statistics live in src/papertrail/drift.py (PSI, calendar-dependent features, the retraining rule; unit-tested) and the
page in src/papertrail/drift_page.py. Evidently 0.7 adds the full interactive reports, linked from the page.

Inputs (attach as Kaggle inputs; the same ones as 06-demo-slice):
  papertrail-processed   transactions.parquet
  papertrail-scores      gnn_scores.parquet, xgb_scores.parquet, blend_summary.json, xgb_summary.json, xgb_v2_final_seed0/1/2.json
CPU, Internet ON, DAGSHUB_TOKEN secret attached (optional: without it the notebook still runs). About 10 minutes.
Download drift_docs.zip from the Output tab and unzip it into PaperTrail/ (it fills docs/ and reports/).



In [ ]:
!pip install -q git+https://github.com/abhibarnwal707/PaperTrail.git evidently==0.7.23
import os
from contextlib import nullcontext

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN").strip()   # .strip(): a pasted newline invalidates a token
    import dagshub, mlflow
    dagshub.init(repo_owner="abhibarnwal707", repo_name="papertrail", mlflow=True)
    print("DagsHub logging: ON")
except Exception as e:  # noqa: BLE001
    # A dead or missing token must not stop the notebook: the drift page is the real output, DagsHub logging is bookkeeping.
    print("DagsHub logging is OFF, the notebook will still build the report. Reason:", repr(e)[:300])

    class _NoMlflow:
        def start_run(self, *a, **k): return nullcontext()
        def log_params(self, *a, **k): pass
        def log_metrics(self, *a, **k): pass
        def log_artifact(self, *a, **k): pass
        def set_tags(self, *a, **k): pass

    mlflow = _NoMlflow()


In [ ]:
import time as _t; print("[cell 2: config] started", _t.strftime("%H:%M:%S"))
import datetime as dt
import glob
import json
import os
import shutil
import urllib.request

import numpy as np
import polars as pl
import xgboost as xgb

from papertrail.drift import daily_psi, drift_table, psi, retrain_decision
from papertrail.drift_page import render
from papertrail.features import build_features, fit_categories
from papertrail.graph_features import HISTORY_FEATURES, add_history_features
from papertrail.metrics import pr_auc
from papertrail.split import assert_valid_split, split_summary, temporal_split

KNOWN = {"transactions.parquet": "/kaggle/input/datasets/abhibarnwal707/papertrail-processed/"}
for _n in ("gnn_scores.parquet", "xgb_scores.parquet", "blend_summary.json", "xgb_summary.json",
           "xgb_v2_final_seed0.json", "xgb_v2_final_seed1.json", "xgb_v2_final_seed2.json"):
    KNOWN[_n] = "/kaggle/input/datasets/abhibarnwal707/papertrail-scores/"


def find(name):
    exact = KNOWN.get(name, "") + name
    if os.path.exists(exact):
        return exact
    for pattern in (f"/kaggle/input/datasets/*/*/{name}", f"/kaggle/input/*/{name}", f"/kaggle/input/**/{name}"):
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found under /kaggle/input: attach the dataset that holds it")


OUT = "/kaggle/working/drift"
os.makedirs(f"{OUT}/docs", exist_ok=True)
os.makedirs(f"{OUT}/reports", exist_ok=True)
SAMPLE, DAY_SAMPLE, EV_SAMPLE, TOP, SEED = 200_000, 50_000, 50_000, 20, 0
APP_URL = "https://papertrail-aml.streamlit.app"          # change if you picked another subdomain
REPO_URL = "https://github.com/abhibarnwal707/PaperTrail"
LABELS_URL = "https://raw.githubusercontent.com/abhibarnwal707/PaperTrail/main/data/demo/feature_labels.json"


In [ ]:
import time as _t; print("[cell 3: features] started", _t.strftime("%H:%M:%S"))
df = pl.read_parquet(find("transactions.parquet")).sort(["ts", "txn_id"])
assert_valid_split(split_summary(df))
dfx = add_history_features(df)
train, val, test = temporal_split(dfx)
cats = fit_categories(train)
FEATURES_HISTORY = [c for c in HISTORY_FEATURES if not c.endswith("age_s")]
tab_cols = build_features(train.head(5), cats).columns
feature_names = tab_cols + FEATURES_HISTORY
xs = json.load(open(find("xgb_summary.json")))
assert feature_names == xs["features"], "feature order differs from the trained models"


def features(part):
    """The exact matrix the models see, plus the day (for the daily view)."""
    return pl.concat([build_features(part, cats), part.select(FEATURES_HISTORY), part.select(day=pl.col("ts").dt.strftime("%m-%d"))], how="horizontal")


def sample(part, n):
    return part.sample(n=min(n, part.height), seed=SEED)


ref = features(sample(train, SAMPLE))
cur = features(sample(test, SAMPLE))
days = features(pl.concat([sample(g, DAY_SAMPLE) for _, g in pl.concat([train, val, test]).group_by(pl.col("ts").dt.date(), maintain_order=True)]))

gain = {}
for s in xs["seeds"]:
    b = xgb.Booster()
    b.load_model(find(f"xgb_v2_final_seed{s}.json"))
    for f, v in b.get_score(importance_type="total_gain").items():
        gain[f] = gain.get(f, 0.0) + v
total = sum(gain.values()) or 1.0
importance = {f: v / total for f, v in gain.items()}

try:
    labels = {k: v["label"] for k, v in json.load(urllib.request.urlopen(LABELS_URL, timeout=20)).items()}
except Exception as e:  # noqa: BLE001
    print("plain-language labels unavailable, using feature names:", repr(e)[:120])
    labels = {}
print(f"{len(feature_names)} features; reference {ref.height:,} rows, current {cur.height:,} rows, daily view {days.height:,} rows over {days['day'].n_unique()} days")


In [ ]:
import time as _t; print("[cell 4: feature drift] started", _t.strftime("%H:%M:%S"))
table = drift_table(ref, cur, feature_names, importance)
daily = daily_psi(ref, days, [f for f in table["feature"].to_list()[:TOP]])
decision = retrain_decision(table)
pl.Config.set_tbl_rows(30)
print(table.select("feature", "importance", "psi", "band", "calendar_dependent").head(TOP))
print("decision:", decision)


In [ ]:
import time as _t; print("[cell 5: prediction drift] started", _t.strftime("%H:%M:%S"))
R = json.load(open(find("blend_summary.json")))
REC, ENS = R["blend_recipe"], R["ensemble"]
THR = {m: ENS[m]["threshold"] for m in ("xgb", "gnn", "blend")}
sc = (pl.read_parquet(find("xgb_scores.parquet")).select("txn_id", "split", "label", "xgb_mean")
        .join(pl.read_parquet(find("gnn_scores.parquet")).select("txn_id", "gnn_mean"), on="txn_id")
        .join(df.select("txn_id", day=pl.col("ts").dt.strftime("%m-%d")), on="txn_id"))


def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))


xz = (logit(sc["xgb_mean"].to_numpy()) - REC["xgb_mu"]) / REC["xgb_sd"]
gz = (logit(sc["gnn_mean"].to_numpy()) - REC["gnn_mu"]) / REC["gnn_sd"]
sc = sc.with_columns(pl.Series("xgb_z", xz), pl.Series("gnn_z", gz), pl.Series("blend_z", REC["w_gnn"] * gz + (1 - REC["w_gnn"]) * xz))
sv, st = sc.filter(pl.col("split") == "val"), sc.filter(pl.col("split") == "test")
score_psi = {m: psi(sv[f"{m}_z"].to_numpy(), st[f"{m}_z"].to_numpy()) for m in ("xgb", "gnn", "blend")}
alert_rate = [{"window": w, **{m: float((part[f"{m}_z"] >= THR[m]).mean()) for m in ("xgb", "gnn", "blend")}}
              for w, part in (("validation, Sep 7-8", sv), ("test, Sep 9-10", st))]
perf_by_day = []
for (day,), part in sc.partition_by("day", as_dict=True, maintain_order=True).items():
    y = part["label"].to_numpy()
    perf_by_day.append({"day": day, "window": part["split"][0], "n": part.height, "positives": int(y.sum()),
                        "alert_rate": float((part["blend_z"] >= THR["blend"]).mean()), "pr_auc": pr_auc(y, part["blend_z"].to_numpy()) if y.sum() else float("nan")})
perf_by_day.sort(key=lambda r: r["day"])
print("score PSI val -> test:", {k: round(v, 4) for k, v in score_psi.items()})
print(pl.DataFrame(alert_rate))
print(pl.DataFrame(perf_by_day))


In [ ]:
import time as _t; print("[cell 6: evidently] started", _t.strftime("%H:%M:%S"))
top_feats = table["feature"].to_list()[:TOP]
ev_files = {}
try:
    from evidently import Report
    from evidently.presets import DataDriftPreset

    def ev_report(reference, current, path):
        snap = Report([DataDriftPreset(method="psi", threshold=0.2)]).run(current_data=current, reference_data=reference)
        snap.save_html(path)
        return os.path.getsize(path) / 1e6

    mb = ev_report(sample(ref, EV_SAMPLE).select(top_feats).to_pandas(), sample(cur, EV_SAMPLE).select(top_feats).to_pandas(), f"{OUT}/docs/feature_drift.html")
    ev_files["Full feature report (Evidently)"] = "feature_drift.html"
    print(f"feature_drift.html {mb:.1f} MB")
    zc = ["xgb_z", "gnn_z", "blend_z"]
    mb = ev_report(sample(sv, EV_SAMPLE).select(zc).to_pandas(), sample(st, EV_SAMPLE).select(zc).to_pandas(), f"{OUT}/docs/prediction_drift.html")
    ev_files["Full prediction report (Evidently)"] = "prediction_drift.html"
    print(f"prediction_drift.html {mb:.1f} MB")
except Exception as e:  # noqa: BLE001
    # The summary page does not depend on Evidently: it is built from papertrail.drift. Say so and carry on.
    print("Evidently report skipped:", repr(e)[:300])


In [ ]:
import time as _t; print("[cell 7: write] started", _t.strftime("%H:%M:%S"))
summary = {
    "generated": dt.date.today().isoformat(),
    "reference": f"train window Sep 1-6 ({ref.height:,}-row sample; validation Sep 7-8 for the scores)",
    "current": f"test window Sep 9-10 ({cur.height:,}-row sample)",
    "decision": decision, "score_psi": score_psi, "alert_rate": alert_rate,
    "perf_by_day": [r for r in perf_by_day if r["window"] == "test"],
    "links": {**ev_files, "Live demo": APP_URL, "Code": REPO_URL},
}
page = render(summary, table, daily, labels, top=TOP)
open(f"{OUT}/docs/index.html", "w").write(page)
open(f"{OUT}/docs/.nojekyll", "w").write("")                       # serve the files as they are
table.write_csv(f"{OUT}/reports/drift_psi.csv")
daily.write_csv(f"{OUT}/reports/drift_daily.csv")
json.dump({**{k: v for k, v in summary.items() if k != "links"}, "perf_by_day_all": perf_by_day,
           "n_features": table.height, "bands": table.group_by("band", "calendar_dependent").len().to_dicts()},
          open(f"{OUT}/reports/drift_summary.json", "w"), indent=1, default=str)
for dp, _, fs in os.walk(OUT):
    for f in sorted(fs):
        print(f"{os.path.relpath(os.path.join(dp, f), OUT):32s} {os.path.getsize(os.path.join(dp, f)) / 1e6:7.2f} MB")
shutil.make_archive("/kaggle/working/drift_docs", "zip", OUT)

with mlflow.start_run(run_name="drift_report"):
    mlflow.log_params({"reference": "train Sep 1-6", "current": "test Sep 9-10", "sample": SAMPLE, "rule": decision["rule"]})
    mlflow.log_metrics({"retrain": float(decision["retrain"]), "n_drifted_stable": float(len(decision["drifted"])),
                        "n_calendar_drift": float(len(decision["expected_calendar_drift"])),
                        **{f"score_psi_{k}": v for k, v in score_psi.items()},
                        **{f"psi_{r['feature']}": r["psi"] for r in table.head(TOP).to_dicts()}})
    for f in ("docs/index.html", "reports/drift_psi.csv", "reports/drift_summary.json"):
        mlflow.log_artifact(f"{OUT}/{f}")
print("Verdict:", "RETRAIN" if decision["retrain"] else "no retraining needed", "| drifted calendar-stable features:", decision["drifted"])
print("Download drift_docs.zip from the Output tab and unzip it into PaperTrail/ (fills docs/ and reports/)")
